# Encoding categories

Most models compute with numbers, so a column of categories must become one or more numerical columns before they can use it. The encoding is not neutral: the numbers tell the model what the categories are, whether they have an order, how far apart they are. A wrong encoding tells it something false, and some models believe it.

## A small table

The fuel and the condition of six cars: a category without an order and one with an order ([Kinds of features](deep-dives/kinds_of_features.ipynb)):

In [ ]:
import pandas as pd

cars = pd.DataFrame({
    "fuel": ["diesel", "petrol", "diesel", "electric", "petrol", "hybrid"],
    "condition": ["good", "excellent", "fair", "excellent", "good", "poor"],
})
cars

,fuel,condition
0,diesel,good
1,petrol,excellent
2,diesel,fair
3,electric,excellent
4,petrol,good
5,hybrid,poor


## Ordinal encoding

The simplest encoding gives each category a whole number. scikit-learn's `OrdinalEncoder` does it, learning the categories with `fit`. Unless told otherwise, it numbers them in alphabetical order:

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

OrdinalEncoder().set_output(transform="pandas").fit_transform(cars).assign(
    fuel_text=cars["fuel"], condition_text=cars["condition"])

,fuel,condition,fuel_text,condition_text
0,0.0,2.0,diesel,good
1,3.0,0.0,petrol,excellent
2,0.0,1.0,diesel,fair
3,1.0,0.0,electric,excellent
4,3.0,2.0,petrol,good
5,2.0,3.0,hybrid,poor


For the condition, alphabetical order is nonsense: "excellent" is 0, "poor" is 3. The order has to be given explicitly, from worst to best:

In [ ]:
condition_encoder = OrdinalEncoder(categories=[["poor", "fair", "good", "excellent"]]).set_output(transform="pandas")
condition_encoder.fit_transform(cars[["condition"]]).assign(condition_text=cars["condition"])

,condition,condition_text
0,2.0,good
1,3.0,excellent
2,1.0,fair
3,3.0,excellent
4,2.0,good
5,0.0,poor


Now 0 is poor and 3 is excellent. The encoding tells the model two things: that the conditions are ordered, which is true, and that the steps between them are equal (poor to fair is as big a step as good to excellent), which is an assumption.

For the fuel, any order is false. Encoded as diesel 0, electric 1, hybrid 2, petrol 3, a linear model would read electric as halfway between diesel and hybrid, and petrol as three times as far from diesel as electric. A category without an order needs another encoding, at least for models that treat numbers as quantities.

scikit-learn also has a `LabelEncoder`. Despite its name it is not for features: it encodes the **target** of a classifier, a single column, always in alphabetical order, with none of `OrdinalEncoder`'s options. In a preprocessing pipeline, use `OrdinalEncoder`.

## One-hot encoding

**One-hot encoding** turns one column of categories into one column per category, with a 1 in the column of the row's category and a 0 in the others: only one column is "hot" in each row. The new columns are sometimes called *dummy* variables:

In [ ]:
from sklearn.preprocessing import OneHotEncoder

fuel_encoder = OneHotEncoder(sparse_output=False).set_output(transform="pandas")
fuel_encoder.fit_transform(cars[["fuel"]]).assign(fuel=cars["fuel"])

,fuel_diesel,fuel_electric,fuel_hybrid,fuel_petrol,fuel
0,1.0,0.0,0.0,0.0,diesel
1,0.0,0.0,0.0,1.0,petrol
2,1.0,0.0,0.0,0.0,diesel
3,0.0,1.0,0.0,0.0,electric
4,0.0,0.0,0.0,1.0,petrol
5,0.0,0.0,1.0,0.0,hybrid


No order and no distances: every pair of fuels is equally different. The cost is one column per category.

By default, `OneHotEncoder` returns a *sparse* matrix, which stores only the positions of the ones. With many categories almost every value is 0, and a sparse matrix saves most of the memory; `sparse_output=False` gives an ordinary table, easier to look at.

### Why not `pandas.get_dummies`

pandas' `get_dummies` does the same in one call, but it learns nothing: it creates a column for each category present *in the data it is given*. New data with fewer fuels gets fewer columns:

In [ ]:
new_cars = pd.DataFrame({"fuel": ["petrol", "diesel"]})
pd.get_dummies(new_cars["fuel"])

,diesel,petrol
0,False,True
1,True,False


Two columns instead of four, and a model trained on four columns can't use them. `OneHotEncoder` remembers the categories it saw in `fit`, so new data always gets the same columns:

In [ ]:
fuel_encoder.transform(new_cars)

,fuel_diesel,fuel_electric,fuel_hybrid,fuel_petrol
0,0.0,0.0,0.0,1.0
1,1.0,0.0,0.0,0.0


### Categories seen only in new data

A car running on liquefied petroleum gas, a fuel the encoder never saw. By default, `OneHotEncoder` refuses it:

In [ ]:
unknown_fuel = pd.DataFrame({"fuel": ["lpg"]})
try:
    fuel_encoder.transform(unknown_fuel)
except ValueError as error:
    print(error)

Found unknown categories ['lpg'] in column 0 during transform


An error is a good default: it says that something new has appeared. When new categories are expected, `handle_unknown="ignore"` encodes them as a row of zeros, "none of the known categories":

In [ ]:
tolerant_fuel_encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore").set_output(transform="pandas")
tolerant_fuel_encoder.fit(cars[["fuel"]]).transform(unknown_fuel)

,fuel_diesel,fuel_electric,fuel_hybrid,fuel_petrol
0,0.0,0.0,0.0,0.0


### Dropping a column

The one-hot columns of a category always add up to 1, so any one of them can be computed from the others. For a linear regression without regularisation, this repeats the information of the intercept, and the weights no longer have a unique solution (the "dummy variable trap"). `drop="first"` removes one column per feature, and `drop="if_binary"` only for categories with two values. Trees and regularised models (as are scikit-learn's logistic regression and most models used in practice) don't need it ([Regularisation](../supervised-learning/ml-fundamentals/regularization.ipynb)). Note that with a dropped column, an unknown category and the dropped one are both encoded as zeros: they become indistinguishable.

## A real table: income in the 1994 census

The *Adult* dataset: 48,842 people from the 1994 United States census, with their age, education, occupation, family situation and country of origin. The target is whether their income was above 50,000 dollars a year. OpenML publishes it; scikit-learn downloads it the first time:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

adult = fetch_openml("adult", version=2, as_frame=True, data_home="data").frame
X = adult.drop(columns="class")
y = (adult["class"] == ">50K").astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
categorical = X.select_dtypes("category").columns.to_list()
X_train[categorical].nunique()

workclass          8
education         16
marital-status     7
occupation        14
relationship       6
race               5
sex                2
native-country    41
dtype: int64

Eight categorical features, with between 2 and 41 categories. Two of them show the two hard cases: an ordered category, and a category with many values.

### An ordered category, already encoded

The education, against the column `education-num` that comes with it:

In [ ]:
X_train.groupby("education", observed=True)["education-num"].agg(["min", "max", "size"]).sort_values("min")

,min,max,size
education,,,
Preschool,1,1,69
1st-4th,2,2,204
5th-6th,3,3,399
7th-8th,4,4,756
9th,5,5,602
10th,6,6,1118
11th,7,7,1450
12th,8,8,527
HS-grad,9,9,12689


Every level of education has exactly one number, from preschool (1) to doctorate (16), in the order of the levels: `education-num` is the ordinal encoding of `education`, with the order given by hand. A model should get one of the two, not both, since they are the same information. As a number, it assumes equal steps: going from 9th to 10th grade counts as much as from a bachelor's degree to a master's. If that is doubtful, one-hot encoding the education drops the assumption, at the price of also dropping the order.

### A category with many values

In [ ]:
country_counts = X_train["native-country"].value_counts()
country_counts.head(3), country_counts.tail(3)

(native-country
 United-States    35101
 Mexico             763
 Philippines        243
 Name: count, dtype: int64,
 native-country
 Laos                  17
 Hungary               16
 Holand-Netherlands     1
 Name: count, dtype: int64)

Of 39,073 training rows, about 35,000 are from the United States. Many countries have a few dozen rows, and the Netherlands has one. One-hot encoding gives every country a column, most of them almost always 0, and a model can't learn anything reliable about a country from one row, or from seventeen.

`OneHotEncoder` can group the rare categories in one *infrequent* column, either below a minimum number of rows (`min_frequency`) or beyond a maximum number of columns (`max_categories`). `handle_unknown="infrequent_if_exist"` sends categories never seen in training to the same column, which is the natural place for them:

In [ ]:
country_encoder = OneHotEncoder(min_frequency=50, handle_unknown="infrequent_if_exist", sparse_output=False)
country_encoder.fit(X_train[["native-country"]])
len(country_encoder.categories_[0]), len(country_encoder.get_feature_names_out())

(42, 24)

42 categories (the 41 countries and the missing value, which the encoder treats as one more category) become 24 columns: one for each of the 22 countries with at least 50 training rows, one for the missing value, and one for all the others.

### Target encoding *(deep dive)*

**Target encoding** replaces each category with the average of the target over the training rows of that category: here, the share of people from each country earning above 50,000 dollars. One column, however many categories, and a number that means something for the task.

The average of a category with few rows is unreliable, so it is pulled towards the average of all rows, the more the fewer rows the category has. With `smooth=20`, a category with $n$ rows gets $(n \cdot \text{its average} + 20 \cdot \text{the overall average}) / (n + 20)$:

In [ ]:
from sklearn.preprocessing import TargetEncoder
from sklearn.model_selection import StratifiedKFold

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)  # the cross-fitting folds, fixed so that results repeat

country_target_encoder = TargetEncoder(smooth=20, cv=folds).fit(X_train[["native-country"]], y_train)
encoded = pd.Series(country_target_encoder.encodings_[0], index=country_target_encoder.categories_[0])
y_train.groupby(X_train["native-country"], observed=True).agg(rows="size", share_above_50k="mean").assign(
    encoded=encoded).sort_values("rows").head(6).round(3)

,rows,share_above_50k,encoded
native-country,,,
Holand-Netherlands,1,0.000,0.228
Hungary,16,0.375,0.300
Laos,17,0.118,0.183
Honduras,17,0.059,0.156
Cambodia,19,0.368,0.302
Scotland,19,0.158,0.200


In [ ]:
y_train.mean().round(3)

np.float64(0.239)

The one person from the Netherlands earned less than 50,000 dollars, an average of 0; the encoding gives 0.228, close to the overall 0.239, because one row says almost nothing. Countries with a few dozen rows are pulled part of the way. The default, `smooth="auto"`, estimates the amount of pulling from the data; it pulls nothing for a category whose rows all have the same target, as a category of one row always does, so a fixed value can be safer when there are many tiny categories.

Target encoding uses the target, and that makes it dangerous. A row encoded with an average that includes its own target carries a piece of the answer: with a category of one row and no smoothing, the encoding *is* the answer. To prevent it, `TargetEncoder.fit_transform` on the training rows uses **cross-fitting**: it splits the rows into folds and encodes each row with the averages of the other folds. So `fit_transform` gives different values from `fit` followed by `transform`, which encodes every row with averages that include it:

In [ ]:
import numpy as np

cross_fitted = TargetEncoder(smooth=20, cv=folds).fit_transform(X_train[["native-country"]], y_train)
not_cross_fitted = TargetEncoder(smooth=20, cv=folds).fit(X_train[["native-country"]], y_train).transform(X_train[["native-country"]])
np.allclose(cross_fitted, not_cross_fitted)

False

Inside a pipeline this is handled correctly: `Pipeline.fit` calls `fit_transform`, and prediction calls `transform` with the encodings learned from all the training rows. Calling `fit` and then `transform` on the training rows by hand is the mistake. [Where data preparation ends](where_preparation_ends.ipynb) shows how large a leak it makes.

## Which encoding, for which model

Three encodings of the seven unordered categorical features (the education is left out, since `education-num` already encodes it), each with a linear model, a logistic regression, and with gradient boosting, a model made of trees. Each pipeline fills the missing categories with the most frequent one and scales the numerical features; 5-fold cross-validation on the training rows:

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

numerical = X.select_dtypes("number").columns.to_list()
nominal = [column for column in categorical if column != "education"]
encoders = {
    "one-hot": OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    "ordinal": OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
    "target": TargetEncoder(cv=folds),
}
models = {
    "logistic regression": LogisticRegression(max_iter=2000),
    "gradient boosting": HistGradientBoostingClassifier(random_state=0),
}


def with_encoder(encoder, model):
    preprocessing = make_column_transformer(
        (make_pipeline(SimpleImputer(strategy="most_frequent"), encoder), nominal),
        (StandardScaler(), numerical),
    )
    return make_pipeline(preprocessing, model)


pd.DataFrame({
    model_name: {
        encoder_name: cross_val_score(with_encoder(encoder, model), X_train, y_train, cv=5).mean()
        for encoder_name, encoder in encoders.items()
    }
    for model_name, model in models.items()
}).round(3)

,logistic regression,gradient boosting
one-hot,0.851,0.873
ordinal,0.824,0.873
target,0.847,0.872


Small differences mean little: the folds alone move each accuracy a little.

- **The logistic regression loses almost three points with ordinal codes.** It multiplies each feature by a weight, so it reads the codes as quantities: occupation 7 counts as seven times occupation 1. One-hot encoding gives every category its own weight.
- **Gradient boosting doesn't care.** A tree splits a feature at thresholds (code ≤ 3, code > 3), and with several splits it can isolate any group of codes, so the false order costs it almost nothing. Ordinal encoding is a common and cheap choice for trees.
- **Target encoding**, which replaces each category with the average target of its training rows (the deep-dive section above), is close to one-hot for both, with a single column per feature.

| Kind of category | Linear models, k-NN, SVMs, networks | Trees and gradient boosting |
| --- | --- | --- |
| Without an order, few categories | One-hot | One-hot or ordinal |
| Without an order, many categories | One-hot grouping the rare ones; target encoding | Ordinal, target encoding, or native support |
| With an order | Ordinal with the order given, or one-hot if the steps are clearly unequal | Ordinal with the order given |

Gradient-boosting libraries can also take categories directly, with no encoding at all: scikit-learn's `HistGradientBoostingClassifier`, which by default reads the columns of pandas' `category` type as categories, and XGBoost with `enable_categorical=True` ([XGBoost](../supervised-learning/algorithms/tree-based/xgboost.ipynb)). Neural networks often learn an **embedding** for each category instead: a short vector of numbers, learned during training, that places similar categories close together (Neural networks *(coming)*).